## 1. Загрузка данных, настройки и общие функции

In [3]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score

# ============================================================
# 1. НАСТРОЙКИ, ЗАГРУЗКА И ОБЩИЕ ФУНКЦИИ
# ============================================================

RANDOM_SEED = 42
ROOT = Path(".")
DATA_DIR = ROOT / "data"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Папка {DATA_DIR.resolve()} не найдена. "
    )

CAT_FEATURES = [
    "lead_source",
    "call_center",
    "region",
    "car_segment",
    "lead_channel",
    "user_tenure_bucket",
    "price_bucket",
    "assignment_hour",
    "assignment_weekday",
    "is_weekend",
]

DROP_COLS = [
    "lead_id",
    "user_id",
    "assignment_ts",
    "assignment_date",
    "target",
]


def load_base_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Загружает train/test и приводит даты и категории к правильному типу."""
    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")

    for frame in (train_df, test_df):
        frame["assignment_ts"] = pd.to_datetime(
            frame["assignment_ts"], errors="raise"
        )
        frame["assignment_date"] = pd.to_datetime(
            frame["assignment_date"], errors="raise"
        ).dt.normalize()

        for column in CAT_FEATURES:
            if column not in frame.columns:
                raise KeyError(f"В данных отсутствует категориальный признак: {column}")
            frame[column] = frame[column].fillna("__MISSING__").astype(str)

    # Проверяем ключи, чтобы merge и submission не портили строки.
    assert train_df["lead_id"].is_unique, "lead_id в train должен быть уникальным"
    assert test_df["lead_id"].is_unique, "lead_id в test должен быть уникальным"
    assert set(train_df["lead_id"]).isdisjoint(set(test_df["lead_id"])), (
        "lead_id из train и test пересекаются"
    )

    # Train сортируем по времени. Порядок test сохраняем исходным.
    train_df = train_df.sort_values(
        ["assignment_ts", "lead_id"]
    ).reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    return train_df, test_df


def daily_average_precision(
    validation_frame: pd.DataFrame,
    scores: np.ndarray,
) -> float:
    """
    Считает официальную локальную метрику:
    AP отдельно для каждой assignment_date, затем среднее по дням.
    """
    evaluation = validation_frame[["assignment_date", "target"]].copy()
    evaluation["score"] = np.asarray(scores, dtype=float)

    if len(evaluation) != len(scores):
        raise ValueError("Количество прогнозов не совпадает с числом строк")

    daily_scores: list[float] = []

    for _, day_frame in evaluation.groupby("assignment_date", sort=True):
        y_true = day_frame["target"].to_numpy()
        y_score = day_frame["score"].to_numpy()

        # В этих данных обычно есть оба класса. Обработка ниже нужна для надежности.
        positives = int(y_true.sum())
        if positives == 0:
            day_ap = 0.0
        elif positives == len(y_true):
            day_ap = 1.0
        else:
            day_ap = average_precision_score(y_true, y_score)

        daily_scores.append(float(day_ap))

    if not daily_scores:
        raise ValueError("Не удалось посчитать Daily AP: валидация пуста")

    return float(np.mean(daily_scores))


def make_date_holdout(
    frame: pd.DataFrame,
    validation_fraction: float = 0.20,
) -> tuple[pd.Series, pd.Series, np.ndarray]:
    """
    Делит train по ЦЕЛЫМ датам.
    Последние примерно 20% уникальных дней идут в validation.
    """
    unique_dates = np.array(
        sorted(frame["assignment_date"].dropna().unique())
    )

    if len(unique_dates) < 2:
        raise ValueError("Для time-based split нужно минимум две разные даты")

    validation_days_count = max(
        1,
        int(np.ceil(len(unique_dates) * validation_fraction)),
    )
    validation_days_count = min(validation_days_count, len(unique_dates) - 1)
    validation_dates = unique_dates[-validation_days_count:]

    train_mask = ~frame["assignment_date"].isin(validation_dates)
    validation_mask = frame["assignment_date"].isin(validation_dates)

    return train_mask, validation_mask, validation_dates


def validate_and_save_submission(
    test_frame: pd.DataFrame,
    scores: np.ndarray,
    output_path: str | Path,
) -> pd.DataFrame:
    """Проверяет формат submission и сохраняет его без индекса."""
    scores = np.asarray(scores, dtype=float)

    submission = pd.DataFrame(
        {
            "lead_id": test_frame["lead_id"].to_numpy(),
            "score": scores,
        }
    )

    assert list(submission.columns) == ["lead_id", "score"]
    assert len(submission) == len(test_frame)
    assert submission["lead_id"].is_unique
    assert submission["lead_id"].tolist() == test_frame["lead_id"].tolist()
    assert np.isfinite(submission["score"]).all()
    assert submission["score"].between(0.0, 1.0).all()

    submission.to_csv(output_path, index=False)
    print(f"Сохранен файл: {output_path}")
    return submission


print("Загрузка train.csv и test.csv...")
train, test = load_base_data(DATA_DIR)
print(f"train: {train.shape}, test: {test.shape}")

Загрузка train.csv и test.csv...
train: (13694, 119), test: (4306, 118)


## 2.baseline CatBoost

In [4]:
# ============================================================
# 2. BASELINE CATBOOST
# ============================================================

baseline_features = [
    column for column in train.columns if column not in DROP_COLS
]

baseline_train_mask, baseline_val_mask, baseline_val_dates = make_date_holdout(
    train,
    validation_fraction=0.20,
)

baseline_train_part = train.loc[baseline_train_mask].copy()
baseline_val_part = train.loc[baseline_val_mask].copy()

print("\nBASELINE")
print(
    "Validation dates:",
    [pd.Timestamp(date).date() for date in baseline_val_dates],
)
print(
    f"Train rows: {len(baseline_train_part)}, "
    f"validation rows: {len(baseline_val_part)}"
)

baseline_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=RANDOM_SEED,
    task_type="CPU",
    allow_writing_files=False,
    verbose=100,
)

baseline_model.fit(
    baseline_train_part[baseline_features],
    baseline_train_part["target"],
    cat_features=CAT_FEATURES,
    eval_set=(
        baseline_val_part[baseline_features],
        baseline_val_part["target"],
    ),
    early_stopping_rounds=100,
    use_best_model=True,
)

baseline_val_predictions = baseline_model.predict_proba(
    baseline_val_part[baseline_features]
)[:, 1]

baseline_global_ap = average_precision_score(
    baseline_val_part["target"],
    baseline_val_predictions,
)
baseline_daily_ap = daily_average_precision(
    baseline_val_part,
    baseline_val_predictions,
)

print(f"Baseline global AP: {baseline_global_ap:.6f}")
print(f"Baseline Daily AP:  {baseline_daily_ap:.6f}")

# tree_count_ уже учитывает early stopping и use_best_model=True.
baseline_final_iterations = max(1, int(baseline_model.tree_count_))
print(f"Деревьев для финальной baseline-модели: {baseline_final_iterations}")

baseline_final_model = CatBoostClassifier(
    iterations=baseline_final_iterations,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    random_seed=RANDOM_SEED,
    task_type="CPU",
    allow_writing_files=False,
    verbose=False,
)

baseline_final_model.fit(
    train[baseline_features],
    train["target"],
    cat_features=CAT_FEATURES,
)

baseline_test_scores = baseline_final_model.predict_proba(
    test[baseline_features]
)[:, 1]

baseline_submission = validate_and_save_submission(
    test,
    baseline_test_scores,
    "catboost_baseline_submission.csv",
)


BASELINE
Validation dates: [datetime.date(2026, 4, 19), datetime.date(2026, 4, 20), datetime.date(2026, 4, 21), datetime.date(2026, 4, 22)]
Train rows: 10272, validation rows: 3422
0:	learn: 0.3553313	test: 0.3263930	best: 0.3263930 (0)	total: 200ms	remaining: 6m 39s
100:	learn: 0.5031259	test: 0.4358915	best: 0.4358915 (100)	total: 4.08s	remaining: 1m 16s
200:	learn: 0.5784839	test: 0.4613295	best: 0.4621098 (195)	total: 7.99s	remaining: 1m 11s
300:	learn: 0.6449576	test: 0.4790587	best: 0.4794839 (296)	total: 11.9s	remaining: 1m 7s
400:	learn: 0.7136252	test: 0.5073969	best: 0.5077475 (397)	total: 15.7s	remaining: 1m 2s
500:	learn: 0.7662938	test: 0.5212330	best: 0.5212330 (500)	total: 19.5s	remaining: 58.4s
600:	learn: 0.8074766	test: 0.5257348	best: 0.5261759 (598)	total: 23.2s	remaining: 54.1s
700:	learn: 0.8443448	test: 0.5296996	best: 0.5307073 (684)	total: 26.9s	remaining: 49.8s
800:	learn: 0.8749612	test: 0.5293247	best: 0.5308260 (726)	total: 30.5s	remaining: 45.6s
Stopped b

## 3. Признаки из events.csv без временной утечки

In [5]:
# ============================================================
# 3. ПРИЗНАКИ ИЗ EVENTS.CSV
# ============================================================


def build_event_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    events_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Строит признаки только по событиям event_ts < assignment_ts.
    Порядок строк test сохраняется неизменным.
    """
    print("\nЗагрузка и обработка events.csv...")
    events = pd.read_csv(events_path)
    events["event_ts"] = pd.to_datetime(events["event_ts"], errors="coerce")

    train_work = train_df.copy()
    test_work = test_df.copy()

    train_work["_dataset"] = "train"
    test_work["_dataset"] = "test"
    train_work["_row_order"] = np.arange(len(train_work), dtype=np.int64)
    test_work["_row_order"] = np.arange(len(test_work), dtype=np.int64)

    all_leads = pd.concat(
        [train_work, test_work],
        ignore_index=True,
        sort=False,
    )

    lead_times = all_leads[["lead_id", "assignment_ts"]].copy()
    assert lead_times["lead_id"].is_unique

    # Inner merge оставляет только события лидов, присутствующих в train/test.
    events_merged = events.merge(
        lead_times,
        on="lead_id",
        how="inner",
        validate="many_to_one",
    )

    matched_events_count = len(events_merged)

    # Критически важное условие против leakage:
    # берем только события, произошедшие СТРОГО ДО назначения обращения.
    events_valid = events_merged.loc[
        events_merged["event_ts"].notna()
        & (events_merged["event_ts"] < events_merged["assignment_ts"])
    ].copy()

    removed_future_events = matched_events_count - len(events_valid)
    print(f"Событий до назначения: {len(events_valid):,}")
    print(f"Удалено событий после/в момент назначения: {removed_future_events:,}")

    events_valid["hours_before_assign"] = (
        events_valid["assignment_ts"] - events_valid["event_ts"]
    ).dt.total_seconds() / 3600.0

    # Базовые агрегаты по истории событий лида.
    event_aggregates = (
        events_valid.groupby("lead_id")
        .agg(
            event_total_count=("event_type", "size"),
            event_type_nunique=("event_type", "nunique"),
            event_price_log_mean=("item_price_log", "mean"),
            event_price_log_std=("item_price_log", "std"),
            event_src_slot_mean=("src_slot", "mean"),
            event_ctx_seq_nunique=("ctx_seq", "nunique"),
            event_hours_since_last=("hours_before_assign", "min"),
            event_hours_since_first=("hours_before_assign", "max"),
        )
        .reset_index()
    )

    # Количество каждого event_type.
    event_type_counts = pd.crosstab(
        events_valid["lead_id"],
        events_valid["event_type"],
    )
    event_type_counts.columns = [
        f"event_count_type_{str(column)}"
        for column in event_type_counts.columns
    ]
    event_type_counts = event_type_counts.reset_index()

    # Активность в нескольких окнах перед назначением.
    event_windows = [
        (24, "event_count_24h"),
        (72, "event_count_72h"),
        (24 * 7, "event_count_7d"),
    ]

    for max_hours, feature_name in event_windows:
        window_counts = (
            events_valid.loc[
                events_valid["hours_before_assign"] <= max_hours
            ]
            .groupby("lead_id")
            .size()
            .rename(feature_name)
            .reset_index()
        )

        event_aggregates = event_aggregates.merge(
            window_counts,
            on="lead_id",
            how="left",
            validate="one_to_one",
        )

    lead_event_features = event_aggregates.merge(
        event_type_counts,
        on="lead_id",
        how="left",
        validate="one_to_one",
    )

    all_leads = all_leads.merge(
        lead_event_features,
        on="lead_id",
        how="left",
        validate="one_to_one",
    )

    # Отдельный индикатор отсутствия истории.
    all_leads["has_events"] = (
        all_leads["event_total_count"].notna().astype("int8")
    )

    # Только счетчики заполняем нулями.
    # Среднюю цену и другие средние НЕ превращаем в 0: NaN корректно обработает CatBoost.
    count_columns = [
        column
        for column in all_leads.columns
        if column.startswith("event_count_")
    ]
    count_columns += [
        "event_total_count",
        "event_type_nunique",
        "event_ctx_seq_nunique",
    ]
    count_columns = list(dict.fromkeys(count_columns))
    all_leads[count_columns] = all_leads[count_columns].fillna(0)

    # Для recency добавляем индикатор пропуска и устойчивое log-преобразование.
    all_leads["event_hours_since_last_missing"] = (
        all_leads["event_hours_since_last"].isna().astype("int8")
    )

    capped_recency = (
        all_leads["event_hours_since_last"]
        .clip(lower=0, upper=24 * 30)
        .fillna(24 * 30)
    )
    all_leads["event_log_hours_since_last"] = np.log1p(capped_recency)

    all_leads["event_history_span_hours"] = (
        all_leads["event_hours_since_first"]
        - all_leads["event_hours_since_last"]
    ).clip(lower=0)
    all_leads["event_history_span_hours"] = (
        all_leads["event_history_span_hours"].fillna(0)
    )

    # Возвращаем train в хронологическом порядке, test — строго в исходном порядке.
    train_enriched_df = (
        all_leads.loc[all_leads["_dataset"].eq("train")]
        .sort_values(["assignment_ts", "lead_id"])
        .drop(columns=["_dataset", "_row_order"])
        .reset_index(drop=True)
    )

    test_enriched_df = (
        all_leads.loc[all_leads["_dataset"].eq("test")]
        .sort_values("_row_order")
        .drop(columns=["_dataset", "_row_order", "target"], errors="ignore")
        .reset_index(drop=True)
    )

    assert len(train_enriched_df) == len(train_df)
    assert len(test_enriched_df) == len(test_df)
    assert test_enriched_df["lead_id"].tolist() == test_df["lead_id"].tolist()

    added_columns = sorted(
        set(train_enriched_df.columns) - set(train_df.columns)
    )
    print(f"Добавлено event-признаков: {len(added_columns)}")

    return train_enriched_df, test_enriched_df


train_enriched, test_enriched = build_event_features(
    train,
    test,
    DATA_DIR / "events.csv",
)


Загрузка и обработка events.csv...
Событий до назначения: 238,581
Удалено событий после/в момент назначения: 16,124
Добавлено event-признаков: 20


## 4. CatBoost на обогащённых данных

In [6]:
# ============================================================
# 4. CATBOOST НА ОБОГАЩЕННЫХ ДАННЫХ
# ============================================================

event_features = [
    column for column in train_enriched.columns if column not in DROP_COLS
]

event_train_mask, event_val_mask, event_val_dates = make_date_holdout(
    train_enriched,
    validation_fraction=0.20,
)

event_train_part = train_enriched.loc[event_train_mask].copy()
event_val_part = train_enriched.loc[event_val_mask].copy()

print("\nCATBOOST + EVENTS")
print(
    "Validation dates:",
    [pd.Timestamp(date).date() for date in event_val_dates],
)
print(f"Количество признаков: {len(event_features)}")

event_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=RANDOM_SEED,
    task_type="CPU",
    allow_writing_files=False,
    verbose=100,
)

event_model.fit(
    event_train_part[event_features],
    event_train_part["target"],
    cat_features=CAT_FEATURES,
    eval_set=(
        event_val_part[event_features],
        event_val_part["target"],
    ),
    early_stopping_rounds=100,
    use_best_model=True,
)

event_val_predictions = event_model.predict_proba(
    event_val_part[event_features]
)[:, 1]

event_global_ap = average_precision_score(
    event_val_part["target"],
    event_val_predictions,
)
event_daily_ap = daily_average_precision(
    event_val_part,
    event_val_predictions,
)

print(f"Events global AP: {event_global_ap:.6f}")
print(f"Events Daily AP:  {event_daily_ap:.6f}")
print(f"Прирост Daily AP против baseline: {event_daily_ap - baseline_daily_ap:+.6f}")

event_final_iterations = max(1, int(event_model.tree_count_))
print(f"Деревьев для финальной event-модели: {event_final_iterations}")

event_final_model = CatBoostClassifier(
    iterations=event_final_iterations,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    random_seed=RANDOM_SEED,
    task_type="CPU",
    allow_writing_files=False,
    verbose=False,
)

event_final_model.fit(
    train_enriched[event_features],
    train_enriched["target"],
    cat_features=CAT_FEATURES,
)

event_test_scores = event_final_model.predict_proba(
    test_enriched[event_features]
)[:, 1]

event_submission = validate_and_save_submission(
    test_enriched,
    event_test_scores,
    "catboost_events_submission.csv",
)


CATBOOST + EVENTS
Validation dates: [datetime.date(2026, 4, 19), datetime.date(2026, 4, 20), datetime.date(2026, 4, 21), datetime.date(2026, 4, 22)]
Количество признаков: 134
0:	learn: 0.4640210	test: 0.4705861	best: 0.4705861 (0)	total: 43.2ms	remaining: 1m 26s
100:	learn: 0.6009006	test: 0.5628886	best: 0.5631188 (99)	total: 3.97s	remaining: 1m 14s
200:	learn: 0.6574346	test: 0.5868537	best: 0.5869145 (198)	total: 7.98s	remaining: 1m 11s
300:	learn: 0.7080118	test: 0.6017907	best: 0.6017907 (300)	total: 11.8s	remaining: 1m 6s
400:	learn: 0.7601559	test: 0.6135764	best: 0.6136361 (391)	total: 15.5s	remaining: 1m 1s
500:	learn: 0.8076731	test: 0.6234084	best: 0.6234084 (500)	total: 19.3s	remaining: 57.8s
600:	learn: 0.8440397	test: 0.6298433	best: 0.6298433 (600)	total: 23.1s	remaining: 53.7s
700:	learn: 0.8744879	test: 0.6344512	best: 0.6349049 (696)	total: 26.8s	remaining: 49.6s
800:	learn: 0.9019709	test: 0.6371522	best: 0.6372954 (798)	total: 31s	remaining: 46.4s
900:	learn: 0.924

## 5. Устойчивые trend-признаки и Optuna с rolling time CV

In [7]:
# ============================================================
# 5. УСТОЙЧИВЫЕ TREND-ФИЧИ + OPTUNA + ROLLING TIME CV
# ============================================================


def add_trend_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Добавляет устойчивые признаки без деления на почти нулевые значения."""
    result = frame.copy()

    def log_activity_ratio(
        recent: pd.Series,
        long_window: pd.Series,
        days: int,
    ) -> pd.Series:
        """
        Логарифм сглаженного отношения:
        активность за короткое окно / средняя дневная активность длинного окна.
        При нулях не создает значения 100000+.
        """
        recent_values = (
            pd.to_numeric(recent, errors="coerce")
            .fillna(0)
            .clip(lower=0)
        )
        long_values = (
            pd.to_numeric(long_window, errors="coerce")
            .fillna(0)
            .clip(lower=0)
        )

        ratio_log = np.log(
            (recent_values + 0.5)
            / (long_values / float(days) + 0.5)
        )
        return ratio_log.clip(-5, 5)

    def smoothed_log_rate(
        numerator: pd.Series,
        denominator: pd.Series,
    ) -> pd.Series:
        numerator_values = (
            pd.to_numeric(numerator, errors="coerce")
            .fillna(0)
            .clip(lower=0)
        )
        denominator_values = (
            pd.to_numeric(denominator, errors="coerce")
            .fillna(0)
            .clip(lower=0)
        )

        rate_log = np.log(
            (numerator_values + 0.5)
            / (denominator_values + 1.0)
        )
        return rate_log.clip(-7, 3)

    result["trend_views_1d_vs_7d"] = log_activity_ratio(
        result["item_views_1d"],
        result["item_views_7d"],
        days=7,
    )

    result["trend_favorites_1d_vs_7d"] = log_activity_ratio(
        result["item_favorites_1d"],
        result["item_favorites_7d"],
        days=7,
    )

    # Разницу цены считаем только там, где история реально существует.
    has_price_history = (
        result["event_price_log_mean"].notna()
        & result["item_price_log"].notna()
    )
    result["price_history_missing"] = (~has_price_history).astype("int8")
    result["price_diff_vs_event_history"] = np.where(
        has_price_history,
        result["item_price_log"] - result["event_price_log_mean"],
        0.0,
    )

    views_90d = (
        pd.to_numeric(result["item_views_90d"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )
    user_age_days = (
        pd.to_numeric(result["user_age_days"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )
    result["log_views_per_day_alive"] = np.log1p(
        (views_90d + 0.5) / (user_age_days + 1.0)
    )

    # Дополнительные устойчивые отношения активности и результата.
    result["contact_rate_7d_log"] = smoothed_log_rate(
        result["user_contacts_7d"],
        result["item_views_7d"],
    )
    result["positive_rate_30d_log"] = smoothed_log_rate(
        result["leadgen_prev_positive_30d"],
        result["leadgen_prev_assigned_30d"],
    )

    # Защита от бесконечных значений после feature engineering.
    numeric_columns = result.select_dtypes(include=[np.number]).columns
    result[numeric_columns] = result[numeric_columns].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    return result


def make_expanding_date_folds(
    frame: pd.DataFrame,
    n_folds: int = 4,
    validation_days: int = 2,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """
    Создает expanding-window folds.
    В каждом следующем fold обучение расширяется вперед по времени,
    validation всегда состоит из более поздних целых дней.
    """
    unique_dates = np.array(
        sorted(frame["assignment_date"].dropna().unique())
    )

    if validation_days < 1:
        raise ValueError("validation_days должен быть >= 1")

    max_possible_folds = len(unique_dates) // validation_days - 1
    if max_possible_folds < 1:
        raise ValueError("Недостаточно дат для rolling time validation")

    actual_folds = min(n_folds, max_possible_folds)
    first_validation_start = (
        len(unique_dates) - actual_folds * validation_days
    )

    folds: list[tuple[np.ndarray, np.ndarray]] = []

    for fold_number in range(actual_folds):
        validation_start = (
            first_validation_start + fold_number * validation_days
        )
        train_dates = unique_dates[:validation_start]
        valid_dates = unique_dates[
            validation_start : validation_start + validation_days
        ]

        if len(train_dates) == 0 or len(valid_dates) == 0:
            continue

        folds.append((train_dates, valid_dates))

    return folds


print("\nДобавление устойчивых trend-признаков...")
train_final = add_trend_features(train_enriched)
test_final = add_trend_features(test_enriched)

final_features = [
    column for column in train_final.columns if column not in DROP_COLS
]

# Параметры поиска. На CPU 20 trials x 4 folds
N_TRIALS = 20
N_FOLDS = 4
VALIDATION_DAYS_PER_FOLD = 2
MAX_ITERATIONS_PER_TRIAL = 1500

rolling_folds = make_expanding_date_folds(
    train_final,
    n_folds=N_FOLDS,
    validation_days=VALIDATION_DAYS_PER_FOLD,
)

print(f"Итоговое количество признаков: {len(final_features)}")
print(f"Количество rolling folds: {len(rolling_folds)}")
for fold_index, (train_dates, valid_dates) in enumerate(rolling_folds, start=1):
    print(
        f"Fold {fold_index}: "
        f"train {pd.Timestamp(train_dates[0]).date()}..{pd.Timestamp(train_dates[-1]).date()}, "
        f"valid {pd.Timestamp(valid_dates[0]).date()}..{pd.Timestamp(valid_dates[-1]).date()}"
    )


def objective(trial: optuna.Trial) -> float:
    params = {
        "iterations": MAX_ITERATIONS_PER_TRIAL,
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.015,
            0.08,
            log=True,
        ),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1.0,
            20.0,
            log=True,
        ),
        "random_strength": trial.suggest_float(
            "random_strength",
            0.05,
            5.0,
            log=True,
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0.0,
            3.0,
        ),
        "loss_function": "Logloss",
        # PRAUC используется только для early stopping внутри CatBoost.
        # Значение Optuna ниже считается именно по Daily AP.
        "eval_metric": "PRAUC",
        "random_seed": RANDOM_SEED,
        "task_type": "CPU",
        "allow_writing_files": False,
        "verbose": False,
    }

    fold_scores: list[float] = []
    fold_tree_counts: list[int] = []

    for train_dates, valid_dates in rolling_folds:
        fold_train_mask = train_final["assignment_date"].isin(train_dates)
        fold_valid_mask = train_final["assignment_date"].isin(valid_dates)

        fold_train = train_final.loc[fold_train_mask]
        fold_valid = train_final.loc[fold_valid_mask]

        model = CatBoostClassifier(**params)
        model.fit(
            fold_train[final_features],
            fold_train["target"],
            cat_features=CAT_FEATURES,
            eval_set=(
                fold_valid[final_features],
                fold_valid["target"],
            ),
            early_stopping_rounds=80,
            use_best_model=True,
            verbose=False,
        )

        fold_predictions = model.predict_proba(
            fold_valid[final_features]
        )[:, 1]

        fold_score = daily_average_precision(
            fold_valid,
            fold_predictions,
        )
        fold_scores.append(fold_score)
        fold_tree_counts.append(int(model.tree_count_))

    mean_score = float(np.mean(fold_scores))
    median_tree_count = max(
        1,
        int(np.median(fold_tree_counts)),
    )

    # Сохраняем число деревьев именно для параметров этого trial.
    trial.set_user_attr("median_tree_count", median_tree_count)
    trial.set_user_attr("fold_daily_ap", [float(x) for x in fold_scores])

    return mean_score


optuna.logging.set_verbosity(optuna.logging.WARNING)
sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
)

print("\nЗапуск Optuna...")
study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
)

print("\nЛУЧШИЙ РЕЗУЛЬТАТ OPTUNA")
print(f"Rolling CV Daily AP: {study.best_value:.6f}")
print("Параметры:")
for parameter_name, parameter_value in study.best_params.items():
    print(f"  {parameter_name}: {parameter_value}")

best_iterations = int(
    study.best_trial.user_attrs["median_tree_count"]
)
print(f"Финальное число деревьев: {best_iterations}")
print(
    "Daily AP по folds лучшего trial:",
    [
        round(value, 6)
        for value in study.best_trial.user_attrs["fold_daily_ap"]
    ],
)

# Не заменяем iterations произвольным значением 1500.
# Используем медианное число деревьев лучшего trial после early stopping.
final_optuna_params = {
    **study.best_params,
    "iterations": best_iterations,
    "loss_function": "Logloss",
    "random_seed": RANDOM_SEED,
    "task_type": "CPU",
    "allow_writing_files": False,
    "verbose": False,
}

optuna_final_model = CatBoostClassifier(**final_optuna_params)
optuna_final_model.fit(
    train_final[final_features],
    train_final["target"],
    cat_features=CAT_FEATURES,
)

optuna_test_scores = optuna_final_model.predict_proba(
    test_final[final_features]
)[:, 1]

optuna_submission = validate_and_save_submission(
    test_final,
    optuna_test_scores,
    "catboost_optuna_trends_submission.csv",
)


Добавление устойчивых trend-признаков...
Итоговое количество признаков: 141
Количество rolling folds: 4
Fold 1: train 2026-04-07..2026-04-14, valid 2026-04-15..2026-04-16
Fold 2: train 2026-04-07..2026-04-16, valid 2026-04-17..2026-04-18
Fold 3: train 2026-04-07..2026-04-18, valid 2026-04-19..2026-04-20
Fold 4: train 2026-04-07..2026-04-20, valid 2026-04-21..2026-04-22

Запуск Optuna...

ЛУЧШИЙ РЕЗУЛЬТАТ OPTUNA
Rolling CV Daily AP: 0.632624
Параметры:
  learning_rate: 0.049248519849344134
  depth: 5
  l2_leaf_reg: 3.3000339465110766
  random_strength: 1.3822747149419152
  bagging_temperature: 1.700560535237379
Финальное число деревьев: 841
Daily AP по folds лучшего trial: [0.58163, 0.640966, 0.674653, 0.633246]
Сохранен файл: catboost_optuna_trends_submission.csv


In [8]:
import pandas as pd

# Загружаем два уже созданных submission-файла
optuna_sub = pd.read_csv("catboost_optuna_trends_submission.csv")
baseline_sub = pd.read_csv("catboost_baseline_submission.csv")

# Проверяем, что лиды идут в одинаковом порядке
assert optuna_sub["lead_id"].equals(baseline_sub["lead_id"])

# Преобразуем вероятности в ранги
optuna_rank = optuna_sub["score"].rank(method="average", pct=True)
baseline_rank = baseline_sub["score"].rank(method="average", pct=True)

# 80% веса сильной Optuna-модели и 20% baseline
blend_submission = pd.DataFrame({
    "lead_id": optuna_sub["lead_id"],
    "score": 0.8 * optuna_rank + 0.2 * baseline_rank,
})

# Проверки
assert len(blend_submission) == len(optuna_sub)
assert blend_submission["lead_id"].is_unique
assert blend_submission["score"].notna().all()
assert blend_submission["score"].between(0, 1).all()

# Сохраняем четвертый submission
blend_submission.to_csv(
    "catboost_rank_blend_80_20.csv",
    index=False,
)

print("Создан файл catboost_rank_blend_80_20.csv")

Создан файл catboost_rank_blend_80_20.csv
